# AI Face Images — Group K-Fold Cross Validation (Identity/Video-aware)

งานนี้มี CSV ไฟล์เดียว **ไม่ต้อง merge** แต่ต้องแบ่ง fold แบบระวัง **identity / video leakage**:
รูปหลายรูปอาจมาจากคนเดียวกันหรือวิดีโอเดียวกัน (เช่น หลายเฟรมจาก deepfake video, หรือรูปหลายรูปของคนเดียวใน imdb_wiki)
ถ้าไม่ group ให้ถูก โมเดลอาจ "จำหน้า" แทนที่จะเรียนรู้ลักษณะ fake/real จริงๆ

**การตั้งค่าตามที่ยืนยัน:**
- Group K-Fold โดยดึง group key จาก `Image Path` (identity/video id)
- Stratify ด้วยคอลัมน์ `Target` (real/fake) เท่านั้น
- 10-fold


In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_colwidth", None)


## 1. Config

แก้ `CSV_PATH` ให้ตรงกับตำแหน่งไฟล์จริงบนเซิร์ฟเวอร์
(`IMAGE_ROOT` ใช้เฉพาะถ้าต้องการเช็คว่าไฟล์รูปมีอยู่จริง — ใส่ `None` เพื่อข้ามขั้นตอนนี้ได้)

In [ ]:
CSV_PATH = Path("/path/to/your_ai_face_dataset.csv")   # TODO: แก้ path ให้ตรงจริง
IMAGE_ROOT = None   # เช่น Path("/media/hdd3/AI_Face_imagesV2_root") หรือ None ถ้าไม่เช็คไฟล์

N_SPLITS = 10
RANDOM_STATE = 42


## 2. โหลดข้อมูล

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]  # กันเผื่อมีช่องว่างใน header

print(f"Total rows: {len(df)}")
print(f"Columns   : {list(df.columns)}")
print(f"\nTarget distribution:")
print(df["Target"].value_counts())
df.head()


## 3. สร้าง Group Key จาก Image Path

แต่ละหมวดของข้อมูลมีรูปแบบ path ต่างกัน ทำให้วิธีดึง "identity / video id" ต้องต่างกันไปด้วย:

| หมวด | ตัวอย่าง path | Group key ที่ดึง |
|---|---|---|
| `DMs/DCFACe/{id}/{n}.jpg` | โฟลเดอร์เลขคือ identity | `DCFACe_{id}` |
| `deepfakes/celebdf/.../..._Celeb-synthesis_idA_idB_clip_frame.png` | คู่ id + clip คือวิดีโอเดียวกัน (frame ต่างกัน) | `celebdf_Celeb-synthesis_idA_idB_clip` |
| `deepfakes/celebdf/.../..._YouTube-real_clip_frame.png` | clip คือวิดีโอเดียวกัน | `celebdf_YouTube-real_clip` |
| `deepfakes/ff++/.../..._youtube_vid_frame.png` | vid คือวิดีโอเดียวกัน | `ffpp_youtube_vid` |
| `deepfakes/ff++/.../..._{Method}_srcA_srcB_frame(N).png` (Method เช่น Deepfakes, FaceSwap, Face2Face, NeuralTextures — รองรับชื่อที่มีตัวเลขปนและไฟล์ซ้ำแบบ `(2)`) | คู่ id คือวิดีโอเดียวกัน | `ffpp_{Method}_srcA_srcB` |
| `deepfakes/dfdc/(real|fake)/..._vid_frame.png` | vid คือวิดีโอเดียวกัน | `dfdc_vid` |
| `deepfakes/dfd/(real|fake)/..._actors__scenario__hash_frame.png` | หลายเฟรมจากวิดีโอเดียวกัน (DeepFakeDetection) | `dfd_actors__scenario__hash` |
| `deepfakes/celebdf/real/..._Celeb-real_id{n}_{clip}_frame.png` | คู่ id+clip คือวิดีโอเดียวกัน | `celebdf_Celeb-real_id{n}_{clip}` |
| `DMs/CommercialTools/{tool}/..._{uuid}_....png` (มี UUID ฝังในชื่อไฟล์ เช่น midjourney) | UUID เดียวกัน = generation เดียวกัน | `commercial_{tool}_{uuid}` |
| `DMs/CommercialTools/{tool}/{id}_face...` (ไม่มี UUID) | id ต่อ tool | `commercial_{tool}_{id}` |
| `.../nmXXXXXXX_rmXXXXXXXXXX_{date-like}_YYYY_faceN...` (ที่ไหนก็ได้ ไม่จำกัด folder) | nm id คือตัวคนจาก IMDB | `imdbwiki_nmXXXXXXX` |
| `.../XXXXXXXX_{date-like}_YYYY_faceN...` (ที่ไหนก็ได้ ไม่จำกัด folder, date-like ยอมรับรูปแบบวันที่เพี้ยน/สั้นได้) | เลข id คือตัวคนจาก Wiki-crop — **จับข้าม folder โดยตั้งใจ** เพราะพบว่า `DMs/StableDiffusion1.5`, `DMs/StableDiffusion_Inpainting` ใช้ id ชุดเดียวกับ `Real/imdb_wiki/wiki` (แปลว่าเป็นรูป generate/inpaint จากคนจริงคนเดียวกัน ถ้าไม่ group ข้าม folder จะเกิด identity leakage ข้าม Target class ได้) | `imdbwiki_wiki_XXXXXXXX` |
| อื่นๆ (GANs/ProGAN, StyleGAN, StyleGAN2/SFHQ, stylegan3, taming_transformer_VQGAN, latent_diffusion, Real/FFHQ, AttGAN, Palette, STARGAN, STGAN, MSG_STYLE_GAN, MMD_GAN ฯลฯ) | รูป synthetic ที่ generate อิสระต่อกัน (sequential id/seed เฉยๆ ไม่ใช่ identity) ไม่มี identity ซ้ำ — ตรวจสอบตัวอย่างจริงแล้วว่าไม่มี id ซ้ำ | ใช้ path ทั้งบรรทัดเป็น group เดี่ยว (unique) |

**สำคัญ:** รูปแบบด้านบนอิงจากตัวอย่างข้อมูลจริงที่เจอแล้วเท่านั้น ข้อมูลเต็ม (1.3M แถว) อาจมี pattern อื่นที่ยังไม่เจอ
ให้ตรวจสอบผลลัพธ์ในเซลล์ verification ด้านล่าง (โดยเฉพาะ breakdown ของ `unmatched_unique` ตาม top-level folder) แล้วเพิ่ม regex ใหม่ได้ถ้าเจอ folder ที่ควรมี identity ซ้ำแต่ยังไม่ถูกจับ

In [ ]:
PATTERNS = [
    # (label, regex, group_template)
    ("DCFACe",
     r"DMs/DCFACe/(?P<id>\d+)/",
     "DCFACe_{id}"),

    ("celebdf_synthesis",
     r"deepfakes/celebdf/[^/]+/\d+_(?P<name>Celeb-synthesis_id\d+_id\d+_\d+)_\d+\.",
     "celebdf_{name}"),

    ("celebdf_real_youtube",
     r"deepfakes/celebdf/[^/]+/\d+_(?P<name>YouTube-real_\d+)_\d+\.",
     "celebdf_{name}"),

    ("celebdf_real_celeb",
     r"deepfakes/celebdf/[^/]+/\d+_(?P<name>Celeb-real_id\d+_\d+)_\d+\.",
     "celebdf_{name}"),

    ("ffpp_youtube",
     r"deepfakes/ff\+\+/[^/]+/\d+_(?P<name>youtube_\d+)_\d+(?:\(\d+\))?\.(?:png|jpg|jpeg)$",
     "ffpp_{name}"),

    ("ffpp_manipulated",
     # method name อาจมีตัวเลขปน (เช่น Face2Face) และไฟล์อาจมี "(2)" ต่อท้ายกรณีซ้ำ/แก้ไข
     r"deepfakes/ff\+\+/[^/]+/\d+_(?P<name>[A-Za-z0-9]+_\d+_\d+)_\d+(?:\(\d+\))?\.(?:png|jpg|jpeg)$",
     "ffpp_{name}"),

    ("dfdc",
     r"deepfakes/dfdc/(?:real|fake)/\d+_(?P<id>\d+)_\d+\.",
     "dfdc_{id}"),

    ("dfd",
     r"deepfakes/dfd/(?:real|fake)/\d+_(?P<name>.+)_\d+\.(?:png|jpg|jpeg)$",
     "dfd_{name}"),

    # ไม่จำกัด folder โดยตั้งใจ: id เดียวกันอาจถูกใช้ซ้ำใน tool อื่น
    # (เช่น StableDiffusion1.5 / StableDiffusion_Inpainting ใช้ wiki id เดิม)
    # ต้อง group ให้ตรงกันข้าม folder เพื่อกัน identity leakage ข้าม Target class
    # ส่วนวันที่ยอมให้หลวมกว่าปกติ (digits/./- ความยาว 6-14 ตัวอักษร) เพราะเจอวันที่เพี้ยน
    # ในข้อมูลจริง เช่น "1989-08-019", "11-03-00", "1994-11-18."
    ("imdb_generic",
     r"(?P<id>nm\d+)_rm\d+_[\d\.\-]{6,14}_\d{4}_face",
     "imdbwiki_{id}"),

    ("wiki_generic",
     r"(?<!nm)(?P<id>\d+)_[\d\.\-]{6,14}_\d{4}_face",
     "imdbwiki_wiki_{id}"),

    ("commercial_uuid",
     # midjourney ฯลฯ ฝัง UUID ของ generation ไว้ในชื่อไฟล์ -> UUID เดียวกันคือรูปจาก generation เดียวกัน
     r"DMs/CommercialTools/(?P<tool>[^/]+)/.*?(?P<uuid>[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})",
     "commercial_{tool}_{uuid}"),

    ("commercial_numeric_id",
     r"DMs/CommercialTools/(?P<tool>[^/]+)/(?P<id>\d+)_",
     "commercial_{tool}_{id}"),
]

COMPILED_PATTERNS = [(label, re.compile(pattern), tmpl) for label, pattern, tmpl in PATTERNS]


def extract_group(path: str, row_idx: int):
    for label, regex, tmpl in COMPILED_PATTERNS:
        m = regex.search(path)
        if m:
            return tmpl.format(**m.groupdict()), label
    # ไม่เข้า pattern ไหนเลย -> ถือว่าเป็นภาพ synthetic อิสระ ไม่มี identity ซ้ำ
    return f"unique_{row_idx}", "unmatched_unique"


groups_and_labels = [
    extract_group(path, idx) for idx, path in enumerate(df["Image Path"])
]
df["group_key"] = [g for g, _ in groups_and_labels]
df["group_pattern"] = [lbl for _, lbl in groups_and_labels]

print("จำนวน record แยกตาม pattern ที่จับได้:")
print(df["group_pattern"].value_counts())


## 4. ตรวจสอบผลการดึง Group Key

- ดูตัวอย่างว่า group ไหนมีมากกว่า 1 รูป (ควรเป็นวิดีโอ/คนที่มีหลายเฟรมจริง)
- ดูตัวอย่างแถวที่ `unmatched_unique` เพื่อยืนยันว่าเป็นภาพ synthetic ที่ควรเป็น unique จริง ไม่ใช่ pattern ที่ยังไม่ได้เขียน regex รองรับ

In [ ]:
group_sizes = df.groupby("group_key").size().sort_values(ascending=False)

print("Top 15 group ที่มีจำนวนรูปมากที่สุด:")
print(group_sizes.head(15))

print(f"\nจำนวน group ทั้งหมด : {df['group_key'].nunique()}")
print(f"จำนวนรูปทั้งหมด      : {len(df)}")
print(f"เฉลี่ยรูปต่อ group    : {len(df) / df['group_key'].nunique():.2f}")


In [ ]:
# ตัวอย่างแถวที่ไม่เข้า pattern ไหนเลย -> ควรเป็น GAN/diffusion/FFHQ ที่ generate อิสระ
unmatched = df[df["group_pattern"] == "unmatched_unique"]
print(f"จำนวนแถวที่เป็น unique group (ไม่เข้า pattern): {len(unmatched)}")
print("\nตัวอย่าง path ของกลุ่มนี้ (สุ่มดู 15 แถว) -> เช็คว่าถูกต้องไหมว่าไม่ควรมี identity ซ้ำ:")
unmatched[["Image Path"]].sample(min(15, len(unmatched)), random_state=RANDOM_STATE)


### 4.1 Breakdown ของ unmatched ตาม top-level dataset folder

ข้อมูลเต็มมีขนาดใหญ่ (หลักล้านแถว) การสุ่มดู 15 แถวอาจไม่พอจับทุก pattern
เซลล์นี้แจกแจง `unmatched_unique` ตาม category ของ path เพื่อดูภาพรวมว่าเหลือ folder ไหนบ้าง
และในแต่ละ folder ยังเป็น "synthetic อิสระ" จริงไหม (ไม่ควรมี id/ชื่อไฟล์ซ้ำเป็นชุดๆ)

In [ ]:
def top_level_category(path: str) -> str:
    # ตัด root "/AI_Face_imagesV2/" ออก แล้วเอา 2 ระดับแรกของ path มาเป็น category
    parts = path.strip("/").split("/")
    if parts and parts[0] == "AI_Face_imagesV2":
        parts = parts[1:]
    return "/".join(parts[:2]) if len(parts) >= 2 else "/".join(parts)

unmatched = unmatched.copy()
unmatched["category"] = unmatched["Image Path"].apply(top_level_category)

cat_counts = unmatched["category"].value_counts()
print("จำนวนแถว unmatched แยกตาม category:")
print(cat_counts)

print("\nตัวอย่าง 3 path ต่อ category (เช็คว่าแต่ละ category ควรเป็น unique จริงไหม):")
for cat in cat_counts.index:
    sample_paths = unmatched.loc[unmatched["category"] == cat, "Image Path"].head(3).tolist()
    print(f"\n[{cat}]  (n={cat_counts[cat]})")
    for p in sample_paths:
        print(f"   {p}")


In [ ]:
# ตัวอย่างกลุ่มที่มีหลายรูป -> เช็คว่ารูปในกลุ่มเดียวกันมาจาก identity/วิดีโอเดียวกันจริง
example_group = group_sizes[group_sizes > 1].index[0]
df[df["group_key"] == example_group][["Image Path", "group_key", "group_pattern", "Target"]]


## 5. (Optional) เช็คว่าไฟล์รูปมีอยู่จริง

ข้ามอัตโนมัติถ้า `IMAGE_ROOT` เป็น `None`

In [ ]:
if IMAGE_ROOT is not None:
    def file_exists(rel_path: str) -> bool:
        p = IMAGE_ROOT / rel_path.lstrip("/")
        return p.exists()

    df["has_image"] = df["Image Path"].apply(file_exists)
    missing = df[~df["has_image"]]
    print(f"Missing image files: {len(missing)} / {len(df)}")
    if len(missing) > 0:
        print(missing["Image Path"].head(20))
else:
    print("ข้ามการเช็คไฟล์รูป (IMAGE_ROOT = None)")


## 6. Stratified Group K-Fold

- **group** = `group_key` (identity/วิดีโอ) → กันรูปจากคนหรือวิดีโอเดียวกันไม่ให้แยกไปคนละ fold
- **stratify** = `Target` (real=0 / fake=1) → คุมสัดส่วน real/fake ให้ใกล้เคียงกันทุก fold
- 10-fold ตามที่ยืนยัน

In [ ]:
X = df.index.values
y = df["Target"].values
groups = df["group_key"].values

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

df["fold"] = -1
for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    df.loc[val_idx, "fold"] = fold_idx

print("จำนวน record ต่อ fold:")
print(df["fold"].value_counts().sort_index())


## 7. ตรวจสอบว่าไม่มี identity/video leakage ระหว่าง fold

In [ ]:
leak_found = False
for fold in range(N_SPLITS):
    val_groups = set(df.loc[df["fold"] == fold, "group_key"])
    train_groups = set(df.loc[df["fold"] != fold, "group_key"])
    overlap = val_groups & train_groups
    if overlap:
        leak_found = True
        print(f"Fold {fold}: ⚠️ LEAKAGE พบ group ซ้ำ {len(overlap)} กลุ่ม")
    else:
        print(f"Fold {fold}: OK — validation {len(val_groups)} groups, ไม่มีการซ้ำกับ train")

print("\n✅ ไม่พบ leakage ในทุก fold" if not leak_found else "\n❌ พบ leakage โปรดตรวจสอบ")


## 8. ตรวจสอบสัดส่วน Target และ Intersection ในแต่ละ fold

In [ ]:
target_dist = df.groupby(["fold", "Target"]).size().unstack(fill_value=0)
target_dist["total"] = target_dist.sum(axis=1)
target_dist["fake_ratio"] = (target_dist[1] / target_dist["total"]).round(3) if 1 in target_dist.columns else np.nan
print("การกระจายของ Target ต่อ fold:")
display(target_dist)

# ไม่ได้ stratify ตาม Intersection แต่ดูไว้เพื่อรู้ทัน ว่า fairness subgroup เอียงไปทาง fold ไหนหรือไม่
intersection_dist = df.groupby(["fold", "Intersection"]).size().unstack(fill_value=0)
print("\nการกระจายของ Intersection (gender x skin tone group) ต่อ fold — ไว้ดูเฉยๆ ไม่ได้ควบคุมโดยตรง:")
display(intersection_dist)


## 9. บันทึกผลลัพธ์

In [ ]:
OUTPUT_CSV = CSV_PATH.with_name(CSV_PATH.stem + "_with_folds.csv")
df.to_csv(OUTPUT_CSV, index=False)
print(f"บันทึกไฟล์แล้ว: {OUTPUT_CSV}")
df.head()


---
### หมายเหตุ

- ถ้าพบว่า `unmatched_unique` มีจำนวนมากผิดปกติ หรือเจอ path pattern ใหม่ที่ควรมี identity ซ้ำแต่ไม่ถูกจับกลุ่ม
  ให้เพิ่ม regex ใหม่ใน `PATTERNS` (เซลล์ที่ 3) แล้วรันใหม่ตั้งแต่เซลล์นั้น
- ตรวจสอบแล้วว่า GANs/ProGAN, StyleGAN, StyleGAN2 (SFHQ), stylegan3, taming_transformer_VQGAN, latent_diffusion,
  Real/FFHQ, AttGAN, Palette, STARGAN, STGAN, MSG_STYLE_GAN, MMD_GAN เป็นภาพ synthetic ที่ generate อิสระต่อกันจริง
  (ไม่มี id ซ้ำ) จึงปล่อยให้เป็น unique group ต่อภาพได้อย่างปลอดภัย — ถ้าข้อมูลจริงมีเพิ่มจาก tool เหล่านี้อีกจำนวนมาก
  แนะนำสุ่มตรวจซ้ำอีกครั้งเผื่อมี id ซ้ำที่ตัวอย่างที่ตรวจไม่ครอบคลุม
- ถ้าอยาก stratify ด้วย `Intersection` ร่วมด้วย (คุม fairness ในแต่ละ fold) สามารถเปลี่ยน `y` ในขั้นตอนที่ 6 เป็น
  `df["Target"].astype(str) + "_" + df["Intersection"].astype(str)` ได้ แต่ต้องระวัง — บาง subgroup อาจมีจำนวนน้อยจนแบ่ง 10 fold ไม่ลงตัว
- ตอน train ให้ filter ด้วยคอลัมน์ `fold` ได้เลย เช่น
  `train_df = df[df["fold"] != k]`, `val_df = df[df["fold"] == k]`
